In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Joint Refinement: Si, Bragg + PDF

This example demonstrates a joint refinement of the Si crystal
structure combining Bragg diffraction and pair distribution function
(PDF) analysis. The Bragg experiment uses time-of-flight neutron
powder diffraction data from SEPD at Argonne, while the PDF
experiment uses data from NOMAD at SNS. A single shared Si structure
is refined simultaneously against both datasets.

## Import Library

In [2]:
from easydiffraction import ExperimentFactory
from easydiffraction import Project
from easydiffraction import StructureFactory
from easydiffraction import download_data

## Define Structure

A single Si structure is shared between the Bragg and PDF
experiments. Structural parameters refined against both datasets
simultaneously.

#### Create Structure

In [3]:
structure = StructureFactory.from_scratch(name='si')

#### Set Space Group

In [4]:
structure.space_group.name_h_m = 'F d -3 m'
structure.space_group.it_coordinate_system_code = '1'

#### Set Unit Cell

In [5]:
structure.cell.length_a = 5.42

#### Set Atom Sites

In [6]:
structure.atom_sites.create(
    label='Si',
    type_symbol='Si',
    fract_x=0,
    fract_y=0,
    fract_z=0,
    wyckoff_letter='a',
    adp_iso=0.2,
)

## Define Experiments

Two experiments are defined: one for Bragg diffraction and one for
PDF analysis. Both are linked to the same Si structure.

### Experiment 1: Bragg (SEPD, TOF)

#### Download Data

In [7]:
bragg_data_path = download_data(id=7, destination='data')

Getting data...


Data #7: Si, SEPD (Argonne)


✅ Data #7 downloaded to 'data/ed-7.xye'


#### Create Experiment

In [8]:
bragg_expt = ExperimentFactory.from_data_path(
    name='sepd', data_path=bragg_data_path, beam_mode='time-of-flight'
)

#### Set Instrument

In [9]:
bragg_expt.instrument.setup_twotheta_bank = 144.845
bragg_expt.instrument.calib_d_to_tof_offset = -9.2
bragg_expt.instrument.calib_d_to_tof_linear = 7476.91
bragg_expt.instrument.calib_d_to_tof_quad = -1.54

#### Set Peak Profile

In [10]:
bragg_expt.peak_profile_type = 'jorgensen'
bragg_expt.peak.broad_gauss_sigma_0 = 5.0
bragg_expt.peak.broad_gauss_sigma_1 = 45.0
bragg_expt.peak.broad_gauss_sigma_2 = 1.0
bragg_expt.peak.exp_decay_beta_0 = 0.04221
bragg_expt.peak.exp_decay_beta_1 = 0.00946
bragg_expt.peak.exp_rise_alpha_0 = 0.0
bragg_expt.peak.exp_rise_alpha_1 = 0.5971

⚠️ Switching peak profile type discards existing peak parameters.                                                                 


Peak profile type for experiment 'sepd' changed to


jorgensen


#### Set Background

In [11]:
bragg_expt.background_type = 'line-segment'
for x in range(0, 35000, 5000):
    bragg_expt.background.create(id=str(x), x=x, y=200)

Background type for experiment 'sepd' already set to


line-segment


#### Set Linked Phases

In [12]:
bragg_expt.linked_phases.create(id='si', scale=13.0)

### Experiment 2: PDF (NOMAD, TOF)

#### Download Data

In [13]:
pdf_data_path = download_data(id=5, destination='data')

Getting data...


Data #5: NOM_9999_Si_640g_PAC_50_ff_ftfrgr_up-to-50.gr


✅ Data #5 already present at 'data/ed-5.gr'. Keeping existing file.


#### Create Experiment

In [14]:
pdf_expt = ExperimentFactory.from_data_path(
    name='nomad',
    data_path=pdf_data_path,
    beam_mode='time-of-flight',
    scattering_type='total',
)

⚠️ No uncertainty (sy) column provided. Defaulting to 0.03.                                                                       


#### Set Peak Profile (PDF Parameters)

In [15]:
pdf_expt.peak.damp_q = 0.02
pdf_expt.peak.broad_q = 0.02
pdf_expt.peak.cutoff_q = 35.0
pdf_expt.peak.sharp_delta_1 = 0.001
pdf_expt.peak.sharp_delta_2 = 4.0
pdf_expt.peak.damp_particle_diameter = 0

#### Set Linked Phases

In [16]:
pdf_expt.linked_phases.create(id='si', scale=1.0)

## Define Project

The project object manages the shared structure, both experiments,
and the analysis.

#### Create Project

In [17]:
project = Project()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#### Add Structure

In [18]:
project.structures.add(structure)

#### Add Experiments

In [19]:
project.experiments.add(bragg_expt)
project.experiments.add(pdf_expt)

## Perform Analysis

This section shows the joint analysis process. The calculator is
auto-resolved per experiment: CrysPy for Bragg, PDFfit for PDF.

#### Set Fit Mode and Weights

In [20]:
project.analysis.fit.mode = 'joint'
project.analysis.joint_fit_experiments.create(id='sepd', weight=0.7)
project.analysis.joint_fit_experiments.create(id='nomad', weight=0.3)

#### Plot Measured vs Calculated (Before Fit)

In [21]:
project.display.plotter.plot_meas_vs_calc(expt_name='sepd')

In [22]:
project.display.plotter.plot_meas_vs_calc(expt_name='nomad')

#### Set Fitting Parameters

Shared structural parameters are refined against both datasets
simultaneously.

In [23]:
structure.cell.length_a.free = True
structure.atom_sites['Si'].adp_iso.free = True

Bragg experiment parameters.

In [24]:
bragg_expt.linked_phases['si'].scale.free = True
bragg_expt.instrument.calib_d_to_tof_offset.free = True
bragg_expt.peak.broad_gauss_sigma_0.free = True
bragg_expt.peak.broad_gauss_sigma_1.free = True
bragg_expt.peak.broad_gauss_sigma_2.free = True
for point in bragg_expt.background:
    point.y.free = True

PDF experiment parameters.

In [25]:
pdf_expt.linked_phases['si'].scale.free = True
pdf_expt.peak.damp_q.free = True
pdf_expt.peak.broad_q.free = True
pdf_expt.peak.sharp_delta_1.free = True
pdf_expt.peak.sharp_delta_2.free = True

#### Show Free Parameters

In [26]:
project.analysis.display.free_params()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,si,cell,,length_a,5.42000,,-inf,inf,Å
2,si,atom_site,Si,adp_iso,0.20000,,-inf,inf,Å²
3,sepd,linked_phases,si,scale,13.00000,,-inf,inf,
4,sepd,peak,,gauss_sigma_0,5.00000,,-inf,inf,μs²
5,sepd,peak,,gauss_sigma_1,45.00000,,-inf,inf,μs/Å
6,sepd,peak,,gauss_sigma_2,1.00000,,-inf,inf,μs²/Å²
7,sepd,instrument,,d_to_tof_offset,-9.20000,,-inf,inf,μs
8,sepd,background,0,y,200.00000,,-inf,inf,
9,sepd,background,5000,y,200.00000,,-inf,inf,
10,sepd,background,10000,y,200.00000,,-inf,inf,


#### Run Fitting

In [27]:
project.analysis.fit()
project.analysis.display.fit_results()
project.display.plotter.plot_param_correlations()

Using all experiments 🔬 ['sepd', 'nomad'] for 'joint' fitting


🚀 Starting fit process with 'lmfit (leastsq)'...


📈 Goodness-of-fit (reduced χ²) change:


,iteration,χ²,improvement [%]
1,1,3378.13,
2,23,844.33,75.0% ↓
3,43,267.01,68.4% ↓
4,63,59.98,77.5% ↓
5,83,52.58,12.3% ↓
6,103,51.89,1.3% ↓
7,170,51.87,


<IPython.core.display.Javascript object>

...

⚠️ Red uncertainty: exceeds the fitted value (consider adding constraints)                                                        


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#### Plot Measured vs Calculated (After Fit)

In [28]:
project.display.plotter.plot_meas_vs_calc(expt_name='sepd')

In [29]:
project.display.plotter.plot_meas_vs_calc(expt_name='nomad')